# Module 6: Clothing Category Classification
## CNN Transfer Learning (MobileNetV2), Confusion Matrix & Live Inference

This notebook demonstrates:
1. Transfer Learning architecture for 10-class fashion garment classification.
2. Loading trained weights from `models/category_classifier.keras`.
3. Inspecting test set confusion matrix and accuracy metrics.
4. Running live visual inference on unseen clothing items with top-3 confidence bars.

In [ ]:
import sys
from pathlib import Path

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf

from src.classifier import ClothingClassifier
from src.config import CLASSIFIER_MODEL_PATH, SPLITS_CSV, CONFUSION_MATRIX_PNG, CLOTHING_CATEGORIES

plt.rcParams["figure.figsize"] = (12, 6)

### 1. Initialize Inference Predictor & Load Model

In [ ]:
classifier = ClothingClassifier(model_path=CLASSIFIER_MODEL_PATH)
print(f"Loaded trained model from: {CLASSIFIER_MODEL_PATH}")
classifier.model.summary(line_length=80)

### 2. Confusion Matrix & Performance Metrics

In [ ]:
if CONFUSION_MATRIX_PNG.exists():
    cm_img = Image.open(CONFUSION_MATRIX_PNG)
    plt.figure(figsize=(12, 10))
    plt.imshow(cm_img)
    plt.axis("off")
    plt.title("Category Classifier - Test Set Confusion Matrix", fontsize=14, fontweight="bold")
    plt.show()
else:
    print(f"Confusion matrix plot not found at {CONFUSION_MATRIX_PNG}. Run train_classifier.py to generate.")

### 3. Live Visual Inference on Unseen Test Garments
Predicting category and top-3 probabilities on randomly selected test set items.

In [ ]:
df_splits = pd.read_csv(SPLITS_CSV)
test_df = df_splits[df_splits["split"] == "test"]
sample_items = test_df.sample(4, random_state=42)

fig, axes = plt.subplots(4, 2, figsize=(11, 13), gridspec_kw={"width_ratios": [1, 2]})

for idx, (_, row) in enumerate(sample_items.iterrows()):
    img_path = row["image_path"]
    actual_cat = row["canonical_category"]
    
    # Run prediction
    pred = classifier.predict(img_path, top_k=3)
    pred_cat = pred["top_category"]
    conf = pred["confidence"]
    is_match = (pred_cat == actual_cat)
    
    # Display image
    img = Image.open(img_path)
    axes[idx, 0].imshow(img)
    match_label = "MATCH" if is_match else "DIFF"
    axes[idx, 0].set_title(f"[{match_label}] Actual: {actual_cat}\nPredicted: {pred_cat} ({conf*100:.1f}%)",
                           fontsize=10, fontweight="bold", color="green" if is_match else "darkorange")
    axes[idx, 0].axis("off")
    
    # Bar chart for Top-3 probabilities
    cats = [x["category"] for x in pred["top_k"]][::-1]
    probs = [x["probability"] * 100 for x in pred["top_k"]][::-1]
    bars = axes[idx, 1].barh(cats, probs, color=["lightgray", "skyblue", "dodgerblue"], edgecolor="black", height=0.55)
    axes[idx, 1].set_xlim(0, 110)
    axes[idx, 1].set_xlabel("Confidence (%)")
    axes[idx, 1].set_title("Top-3 Predicted Class Probabilities", fontsize=10, fontweight="bold")
    for bar in bars:
        w = bar.get_width()
        axes[idx, 1].text(w + 2, bar.get_y() + 0.15, f"{w:.1f}%", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()